This notebook allows to search OpenAlex using a specific search phrase, and embed and store the fecthed articles into a chroma database.

In [1]:
import requests


# Fetch articles from OpenAlex

##### Functions for OpenAlex search

In [2]:
import math

#function to count pages and number of results (without fetching)
def get_total_pages(query, per_page=200):
    base_url = "https://api.openalex.org/works"  # Replace with the actual API endpoint
    # Create filters
    filters = [
        "has_abstract:true",
        "has_fulltext:true"
        #f"from_publication_date:{min_year}-01-01"
    ]
    
    params = {
        "search": query,
        "filter": ",".join(filters) if filters else None,
        "per_page": 1,  # Fetch only one result to get metadata
    }
    
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    
    total_results = data.get("meta", {}).get("count", 0)
    total_pages = math.ceil(total_results / per_page)  # Calculate total pages
    
    return total_results, total_pages


# function to parse the text
def reconstruct_text(inverted_index):
    word_index = []
    for k,v in inverted_index.items():
        for index in v:
            word_index.append([k,index])

    word_index = sorted(word_index,key = lambda x : x[1])

    word_list = []
    for i in range(len(word_index)):
        word_list.append(word_index[i][0])

    separator = ' '
    reconstructed_text = separator.join(word_list)

    return reconstructed_text

# function that uses openalex to search web for papers
def search_openalex(search_phrase):
    base_url = "https://api.openalex.org/works"  # Replace with the actual API endpoint

    # Create filters
    filters = [
        "has_abstract:true",
        "has_fulltext:true"
        #f"from_publication_date:{min_year}-01-01"
    ]

    # Construct the query parameters
    params = {
    "search": search_phrase,
    "filter": str.join(",", filters),  # Only return works with abstracts
    "per_page": 200,  # maximum allowed per page ,
    "cursor": "*", 
    }

    #fetch articles from all pages
    all_results = []
    abstract_list = []

    i = 0
    while params["cursor"]:
        print("page " + str(i))
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        res_json = r.json()
        
        # Add the current batch of results
        all_results.extend(res_json.get("results", []))
        
        # Update the cursor to fetch the next page
        params["cursor"] = res_json.get("meta", {}).get("next_cursor")
        i = i+1
        
        for j in range(len(res_json["results"])):
            abstract_list.append(reconstruct_text(res_json["results"][j]['abstract_inverted_index']))

    return all_results, abstract_list

##### OpenAlex Search

In [3]:
search_phrase = '("mCRPC" OR "metastatic CRPC" OR "metastatic castration-resistant prostate cancer" \
OR "metastatic castration-resistant prostate carcinoma")'

In [4]:
#collect 
total_results, total_pages = get_total_pages(search_phrase)
print(f"Total results: {total_results}")
print(f"Total pages: {total_pages}")

res_, abstract = search_openalex(search_phrase)
print(f"Total results fetched: {len(res_)}")

Total results: 10829
Total pages: 55
page 0
page 1
page 2
page 3
page 4
page 5
page 6
page 7
page 8
page 9
page 10
page 11
page 12
page 13
page 14
page 15
page 16
page 17
page 18
page 19
page 20
page 21
page 22
page 23
page 24
page 25
page 26
page 27
page 28
page 29
page 30
page 31
page 32
page 33
page 34
page 35
page 36
page 37
page 38
page 39
page 40
page 41
page 42
page 43
page 44
page 45
page 46
page 47
page 48
page 49
page 50
page 51
page 52
page 53
page 54
page 55
Total results fetched: 10825


In [7]:
res_[0]

{'id': 'https://openalex.org/W3020771563',
 'doi': 'https://doi.org/10.1056/nejmoa1911440',
 'title': 'Olaparib for Metastatic Castration-Resistant Prostate Cancer',
 'display_name': 'Olaparib for Metastatic Castration-Resistant Prostate Cancer',
 'relevance_score': 1755.7739,
 'publication_year': 2020,
 'publication_date': '2020-04-28',
 'ids': {'openalex': 'https://openalex.org/W3020771563',
  'doi': 'https://doi.org/10.1056/nejmoa1911440',
  'mag': '3020771563',
  'pmid': 'https://pubmed.ncbi.nlm.nih.gov/32846073'},
 'language': 'en',
 'primary_location': {'is_oa': True,
  'landing_page_url': 'https://doi.org/10.1056/nejmoa1911440',
  'pdf_url': 'https://www.nejm.org/doi/pdf/10.1056/NEJMoa1911440?articleTools=true',
  'source': {'id': 'https://openalex.org/S62468778',
   'display_name': 'New England Journal of Medicine',
   'issn_l': '0028-4793',
   'issn': ['0028-4793', '1533-4406'],
   'is_oa': False,
   'is_in_doaj': False,
   'is_indexed_in_scopus': True,
   'is_core': True,
   

In [15]:
import csv
# Create a CSV file
with open('titles_and_abstracts.csv', 'w', newline='', encoding='utf-8') as file:
    # Define the writer
    writer = csv.writer(file, delimiter=';')

    # Write the header row
    writer.writerow(["Title", "Abstract"])

    # Write the rows with titles and corresponding abstracts
    for i in range(len(abstract)):
        writer.writerow([res_[i]["title"], abstract[i]])

print("CSV file 'titles_and_abstracts.csv' created successfully.")


CSV file 'titles_and_abstracts.csv' created successfully.


# Embed end store articles in chromaDB

In [16]:
!pip install chromadb
!pip install sentence_transformers

In [17]:
import chromadb
CHROMA_DATA_PATH = "chroma_data2/"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
client.delete_collection("searchable_db_collection")

In [18]:
#create or get collection

import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data2/"
EMBED_MODEL =  "mixedbread-ai/mxbai-embed-large-v1"
COLLECTION_NAME = "searchable_db_collection"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)

existing_collections = client.list_collections()
print(existing_collections)
if COLLECTION_NAME not in [col.name for col in existing_collections]:
    collection = client.create_collection(
                                        name=COLLECTION_NAME,
                                        embedding_function=embedding_func,
                                        metadata={"hnsw:space": "cosine"})
else:
    collection = client.get_collection(name="searchable_db_collection")

[]


In [19]:
#get titles and metadata, create IDs
string_ids = [str(i) for i in range(len(res_))]  # Generating unique string IDs
titles = [item["title"] or "Unknown Title" for item in res_]
metadatas = [
    {
        "titles": item["title"] or "Unknown Title",
        "first_author": (
            item.get("authorships", [{}])[0].get("author", {}).get("display_name")
            if item.get("authorships") and len(item.get("authorships")) > 0
            else "Unknown Author"  # Fallback to "Unknown Author" if authorships is None or empty
        ),  # Fallback to "Unknown Author" if display_name is None
        "journal": (
            (item.get("primary_location", {}).get("source", {}) or {}).get("display_name")
            or "Unknown Journal"
        ),   # Fallback to "Unknown Journal" if display_name is None
        "year": item["publication_year"] or "Unknown Publication Year"
    }
    for item in res_
]

In [20]:
from tqdm import tqdm  # Import the progress bar library

# Define batch size (tuning this can improve performance)
batch_size = 10  # Adjust batch size as needed

# Get the total number of documents
total_docs = len(res_)

# Loop through the data in chunks and add to the collection
for i in tqdm(range(0, total_docs, batch_size), desc="Adding data to collection", unit="batch"):
    # Create a chunk of data
    start_idx = i
    end_idx = min(i + batch_size, total_docs)
    
    documents_chunk = [titles[j] + " " + abstract[j] for j in range(start_idx, end_idx)]
    ids_chunk = string_ids[start_idx:end_idx]
    metadatas_chunk = metadatas[start_idx:end_idx]
    
    # Add the current chunk to the collection
    collection.add(
        documents=documents_chunk,
        ids=ids_chunk,
        metadatas=metadatas_chunk
    )  

Adding data to collection: 100%|██████████| 1083/1083 [6:26:46<00:00, 21.43s/batch]     
